In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df    = pd.read_csv('../data/processed/lst_split.csv')
train = df[df['split'] == 'train'].reset_index(drop=True)

FEATURES = ['NDVI', 'NDBI', 'NDWI', 'frac_tree', 'frac_shrub', 'frac_grass',
            'frac_crop', 'frac_built', 'frac_bare', 'frac_water']
y      = train['LST']
groups = train['block_id']
cv     = GroupKFold(n_splits=5)
SCORING = {'MAE': 'neg_mean_absolute_error', 'RMSE': 'neg_root_mean_squared_error', 'R2': 'r2'}
print('Training rows:', len(train))

Training rows: 46586


In [2]:
param_grid = {
    'learning_rate':    [0.05, 0.1],
    'max_leaf_nodes':   [15, 31, 63],
    'min_samples_leaf': [20, 50],
}
gbm = HistGradientBoostingRegressor(max_iter=300, early_stopping=False, random_state=42)

search = GridSearchCV(gbm, param_grid, cv=cv, scoring='neg_mean_absolute_error',
                      n_jobs=-1, verbose=1)
search.fit(train[FEATURES], y, groups=groups)

print('Best settings:', search.best_params_)
print('Best CV MAE:', round(-search.best_score_, 3))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best settings: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'min_samples_leaf': 20}
Best CV MAE: 1.39


In [3]:
best = HistGradientBoostingRegressor(max_iter=300, early_stopping=False,
                                     random_state=42, **search.best_params_)
r = cross_validate(best, train[FEATURES], y, groups=groups, cv=cv,
                   scoring=SCORING, return_train_score=True, n_jobs=-1)

gbm_cv = {'model': 'Gradient Boosting (tuned)',
          'MAE': -r['test_MAE'].mean(),   'MAE_sd': r['test_MAE'].std(),
          'RMSE': -r['test_RMSE'].mean(), 'RMSE_sd': r['test_RMSE'].std(),
          'R2': r['test_R2'].mean(),      'R2_sd': r['test_R2'].std()}

cv_all = pd.concat([pd.read_csv('../outputs/cv_results.csv'), pd.DataFrame([gbm_cv])],
                   ignore_index=True).round(3)
print(cv_all[['model', 'MAE', 'MAE_sd', 'RMSE', 'R2']])

print('\nGradient Boosting train MAE:', round(-r['train_MAE'].mean(), 3),
      '| validation MAE:', round(-r['test_MAE'].mean(), 3),
      '| gap:', round(-r['test_MAE'].mean() + r['train_MAE'].mean(), 3))

                       model    MAE  MAE_sd   RMSE     R2
0            Baseline (mean)  2.372   0.086  3.098 -0.015
1          Linear Regression  1.472   0.041  1.889  0.619
2      Random Forest (tuned)  1.395   0.028  1.802  0.653
3  Gradient Boosting (tuned)  1.390   0.025  1.791  0.657

Gradient Boosting train MAE: 1.159 | validation MAE: 1.39 | gap: 0.23


In [4]:
test  = df[df['split'] == 'test'].reset_index(drop=True)
preds = pd.read_csv('../data/processed/test_predictions.csv')
assert (preds['cell_id'].values == test['cell_id'].values).all() and \
       (preds['year'].values == test['year'].values).all(), 'Row order mismatch'

best.fit(train[FEATURES], y)
p = best.predict(test[FEATURES])

gbm_test = {'model': 'Gradient Boosting (tuned)',
            'MAE': mean_absolute_error(test['LST'], p),
            'RMSE': np.sqrt(mean_squared_error(test['LST'], p)),
            'R2': r2_score(test['LST'], p)}
test_all = pd.concat([pd.read_csv('../outputs/test_results.csv'), pd.DataFrame([gbm_test])],
                     ignore_index=True).round(3)
print(test_all)

# block bootstrap: Random Forest MAE minus Gradient Boosting MAE
rng = np.random.default_rng(42)
e = pd.DataFrame({'block': test['block_id'],
                  'rf':  (test['LST'] - preds['pred_rf']).abs(),
                  'gbm': (test['LST'] - p).abs()})
sums, counts = e.groupby('block')[['rf', 'gbm']].sum(), e.groupby('block').size()
diffs = []
for _ in range(1000):
    pick = rng.choice(sums.index, size=len(sums), replace=True)
    s, n = sums.loc[pick].sum(), counts.loc[pick].sum()
    diffs.append(s['rf'] / n - s['gbm'] / n)
diffs = np.array(diffs)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'\nRF − GBM MAE: {diffs.mean():.3f} °C  (95% interval {lo:.3f} to {hi:.3f})')
print('Share of resamples where GBM beats RF:', round((diffs > 0).mean(), 3))

                       model    MAE   RMSE     R2
0            Baseline (mean)  2.410  3.153 -0.026
1          Linear Regression  1.462  1.916  0.621
2      Random Forest (tuned)  1.382  1.810  0.662
3  Gradient Boosting (tuned)  1.371  1.790  0.669

RF − GBM MAE: 0.011 °C  (95% interval 0.001 to 0.020)
Share of resamples where GBM beats RF: 0.987


In [5]:
cv_all.to_csv('../outputs/cv_results_with_gbm.csv', index=False)
test_all.to_csv('../outputs/test_results_with_gbm.csv', index=False)
pd.DataFrame([{**search.best_params_, 'rf_minus_gbm_mae': round(diffs.mean(), 3),
               'ci_low': round(lo, 3), 'ci_high': round(hi, 3)}]) \
  .to_csv('../outputs/gbm_summary.csv', index=False)
print('Saved')

Saved
